In [ ]:
"""
Lab 10 & 11: Code Refactoring và Fine-Tuning với Gemini
Chương 10: Refactoring Code with GenAI
Chương 11: Fine-Tuning Models with Gemini
"""
import json
import logging
import os
import numpy as np
from typing import Dict, List, Tuple, Optional

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


# ============================================================
# LAB 10: CODE REFACTORING
# ============================================================

# --- TRƯỚC KHI REFACTOR (Code Smell) ---

def calculate_distance_BEFORE(data: dict) -> dict:
    """
    Hàm gốc – có nhiều vấn đề:
    - Duplicate code (đọc df1, df2 hai lần)
    - print() thay vì logger
    - Nested loop thay vì vectorization
    - Không tách biệt responsibilities
    """
    dist_type = data.get("distance")

    if dist_type == "L1":
        print("Info: computing L1 distance...")
        a = data.get("df1")
        b = data.get("df2")
        dist = np.sum(np.abs(np.array(a) - np.array(b)))
        return {"distance": float(dist)}

    elif dist_type == "L2":
        print("Info: computing L2 distance...")
        a = data.get("df1")
        b = data.get("df2")
        dist_2 = 0
        for i in range(len(a)):
            for j in range(len(a[i])):
                dist_2 += (a[i][j] - b[i][j]) ** 2
        dist = np.sqrt(dist_2)
        return {"distance": float(dist)}

    return {"error": "Unknown distance type"}


# --- SAU KHI REFACTOR (Clean Code) ---

def _parse_matrices(data: dict) -> Tuple[np.ndarray, np.ndarray, str]:
    """
    Đọc và validate tham số từ request data.

    Refactoring: tách logic đọc data ra hàm riêng → tránh duplicate.

    Args:
        data: dict chứa 'df1', 'df2', 'distance'
    Returns:
        Tuple (matrix_a, matrix_b, distance_type)
    Raises:
        KeyError: nếu thiếu key bắt buộc
        ValueError: nếu hai matrix khác shape
    """
    a = np.array(data["df1"])
    b = np.array(data["df2"])
    dist_type = data["distance"]

    if a.shape != b.shape:
        raise ValueError(f"Matrix shape mismatch: {a.shape} vs {b.shape}")

    return a, b, dist_type


def _compute_l1(a: np.ndarray, b: np.ndarray) -> float:
    """
    Tính Manhattan distance (L1 norm) – vectorized.

    Refactoring: tách hàm tính distance riêng, dùng numpy thay print.
    """
    return float(np.sum(np.abs(a - b)))


def _compute_l2(a: np.ndarray, b: np.ndarray) -> float:
    """
    Tính Euclidean distance (L2 norm) – vectorized.

    Refactoring: thay nested loop O(n²) bằng numpy vectorization O(n).
    Performance gain: ~100x nhanh hơn với matrix 100x100.
    """
    return float(np.sqrt(np.sum((a - b) ** 2)))


def calculate_distance_AFTER(data: dict) -> dict:
    """
    Hàm đã refactor – Clean Code:
    - Single responsibility per function
    - DRY: parse_matrices() dùng một lần
    - logger thay vì print
    - Vectorization thay nested loop
    """
    a, b, dist_type = _parse_matrices(data)

    distance_functions = {
        "L1": _compute_l1,
        "L2": _compute_l2,
    }

    if dist_type not in distance_functions:
        raise ValueError(f"Unsupported distance type: {dist_type}")

    logger.info("Computing %s distance", dist_type)
    dist = distance_functions[dist_type](a, b)
    return {"distance": dist}


def benchmark_nested_vs_vectorized(size: int = 100) -> Dict[str, float]:
    """
    Benchmark để minh họa lợi ích của vectorization.

    Args:
        size: kích thước matrix (size x size)
    Returns:
        dict với thời gian thực thi của hai cách
    """
    import time

    a = np.random.rand(size, size)
    b = np.random.rand(size, size)

    start = time.perf_counter()
    dist_2 = 0
    for i in range(len(a)):
        for j in range(len(a[i])):
            dist_2 += (a[i][j] - b[i][j]) ** 2
    nested_time = time.perf_counter() - start

    start = time.perf_counter()
    _ = float(np.sqrt(np.sum((a - b) ** 2)))
    vectorized_time = time.perf_counter() - start

    return {
        "nested_loop_ms": nested_time * 1000,
        "vectorized_ms": vectorized_time * 1000,
        "speedup": nested_time / vectorized_time if vectorized_time > 0 else float('inf')
    }


# ============================================================
# LAB 11: FINE-TUNING VỚI GEMINI
# ============================================================

def create_tuning_example(
    system_prompt: str,
    user_prompt: str,
    assistant_response: str
) -> dict:
    """
    Tạo một tuning example cho Gemini supervised tuning.

    Gemini format: mỗi example gồm text_input và output.
    """
    text_input = f"{system_prompt}\n\n{user_prompt}"
    return {
        "text_input": text_input,
        "output": assistant_response,
    }


def generate_training_dataset() -> List[dict]:
    """
    Tạo tập dữ liệu training cho fine-tuning.

    Mục tiêu: huấn luyện model trả về code sạch (không comment thừa).
    Theo Gemini docs: dùng TuningDataset với nhiều TuningExample.

    Returns:
        Danh sách training examples dạng {'text_input', 'output'}
    """
    system_prompt = (
        "You are a Python code assistant. When given a function signature, "
        "implement it with clean code only. No explanations, no inline comments, "
        "no docstrings. Return only the implementation."
    )

    examples_data = [
        (
            "def add(a: int, b: int) -> int:",
            "def add(a: int, b: int) -> int:\n    return a + b"
        ),
        (
            "def factorial(n: int) -> int:",
            "def factorial(n: int) -> int:\n    if n <= 1:\n        return 1\n    return n * factorial(n - 1)"
        ),
        (
            "def is_palindrome(s: str) -> bool:",
            "def is_palindrome(s: str) -> bool:\n    s = s.lower().replace(' ', '')\n    return s == s[::-1]"
        ),
        (
            "def fibonacci(n: int) -> int:",
            "def fibonacci(n: int) -> int:\n    if n <= 1:\n        return n\n    a, b = 0, 1\n    for _ in range(2, n + 1):\n        a, b = b, a + b\n    return b"
        ),
        (
            "def flatten(lst: list) -> list:",
            "def flatten(lst: list) -> list:\n    result = []\n    for item in lst:\n        if isinstance(item, list):\n            result.extend(flatten(item))\n        else:\n            result.append(item)\n    return result"
        ),
        (
            "def count_words(text: str) -> Dict[str, int]:",
            "def count_words(text: str) -> Dict[str, int]:\n    words = text.lower().split()\n    return {word: words.count(word) for word in set(words)}"
        ),
        (
            "def binary_search(arr: list, target: int) -> int:",
            "def binary_search(arr: list, target: int) -> int:\n    left, right = 0, len(arr) - 1\n    while left <= right:\n        mid = (left + right) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            left = mid + 1\n        else:\n            right = mid - 1\n    return -1"
        ),
        (
            "def reverse_string(s: str) -> str:",
            "def reverse_string(s: str) -> str:\n    return s[::-1]"
        ),
        (
            "def sum_digits(n: int) -> int:",
            "def sum_digits(n: int) -> int:\n    return sum(int(d) for d in str(abs(n)))"
        ),
        (
            "def remove_duplicates(lst: list) -> list:",
            "def remove_duplicates(lst: list) -> list:\n    seen = set()\n    return [x for x in lst if not (x in seen or seen.add(x))]"
        ),
        (
            "def celsius_to_fahrenheit(c: float) -> float:",
            "def celsius_to_fahrenheit(c: float) -> float:\n    return c * 9 / 5 + 32"
        ),
        (
            "def is_prime(n: int) -> bool:",
            "def is_prime(n: int) -> bool:\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True"
        ),
        (
            "def get_quadratic_roots(a: float, b: float, c: float) -> Optional[Tuple[float, float]]:",
            "def get_quadratic_roots(a: float, b: float, c: float) -> Optional[Tuple[float, float]]:\n    discriminant = b**2 - 4*a*c\n    if discriminant < 0:\n        return None\n    sqrt_d = discriminant**0.5\n    return (-b + sqrt_d) / (2*a), (-b - sqrt_d) / (2*a)"
        ),
        (
            "def merge_sorted(a: list, b: list) -> list:",
            "def merge_sorted(a: list, b: list) -> list:\n    result, i, j = [], 0, 0\n    while i < len(a) and j < len(b):\n        if a[i] <= b[j]:\n            result.append(a[i])\n            i += 1\n        else:\n            result.append(b[j])\n            j += 1\n    return result + a[i:] + b[j:]"
        ),
        (
            "def matrix_transpose(matrix: list) -> list:",
            "def matrix_transpose(matrix: list) -> list:\n    return [list(row) for row in zip(*matrix)]"
        ),
    ]

    training_data = []
    for sig, impl in examples_data:
        user_msg = f"FUNCTION: {{{{ {sig} }}}}\nCODE:"
        training_data.append(create_tuning_example(system_prompt, user_msg, impl))

    return training_data


def save_training_jsonl(data: List[dict], filepath: str) -> None:
    """
    Lưu training data vào file JSONL.

    Args:
        data: danh sách training examples
        filepath: đường dẫn file đầu ra
    """
    with open(filepath, 'w', encoding='utf-8') as f:
        for example in data:
            f.write(json.dumps(example, ensure_ascii=False) + '\n')
    print(f"Saved {len(data)} training examples to {filepath}")


def run_gemini_tuning_job(
    training_data: List[dict],
    model: str = "gemini-2.5-flash",
    tuned_model_display_name: str = "python-clean-code-tuned",
    epoch_count: int = 1,
    api_key: Optional[str] = None,
) -> str:
    """
    Khởi chạy supervised tuning job trên Gemini.

    Args:
        training_data: danh sách dict có text_input và output
        model: base model để fine-tune
        tuned_model_display_name: tên hiển thị của model sau tune
        epoch_count: số epoch tuning
        api_key: Gemini API key; nếu None sẽ đọc GEMINI_API_KEY
    Returns:
        Tuning job name
    """
    try:
        from google import genai
        from google.genai import types

        api_key = api_key or os.getenv("GEMINI_API_KEY")
        client = genai.Client(api_key=api_key) if api_key else genai.Client()

        training_dataset = types.TuningDataset(
            examples=[
                types.TuningExample(text_input=item["text_input"], output=item["output"])
                for item in training_data
            ]
        )

        tuning_job = client.tunings.tune(
            base_model=model,
            training_dataset=training_dataset,
            config=types.CreateTuningJobConfig(
                epoch_count=epoch_count,
                tuned_model_display_name=tuned_model_display_name,
            ),
        )

        print(f"Tuning job created: {tuning_job.name}")
        print(f"State: {tuning_job.state}")
        print(f"Check status: client.tunings.get(name='{tuning_job.name}')")

        tuned_endpoint = getattr(getattr(tuning_job, "tuned_model", None), "endpoint", None)
        if tuned_endpoint:
            print(f"Tuned model endpoint: {tuned_endpoint}")

        return tuning_job.name

    except Exception as e:
        return f"[Cần GEMINI_API_KEY và quyền tuning model: {e}]"


# ============================================================
# DEMO
# ============================================================

if __name__ == "__main__":
    print("=" * 60)
    print("LAB 10: Code Refactoring Demo")
    print("=" * 60)

    test_data = {
        "df1": [[1, 2], [3, 4]],
        "df2": [[5, 6], [7, 8]],
        "distance": "L1"
    }

    print("\n--- Test cả hai phiên bản ---")
    result_before = calculate_distance_BEFORE(test_data)
    result_after = calculate_distance_AFTER(test_data)

    print(f"BEFORE (L1): {result_before['distance']}")
    print(f"AFTER  (L1): {result_after['distance']}")
    assert result_before['distance'] == result_after['distance'], "Results should match!"
    print("✓ Kết quả giống nhau – refactoring không thay đổi functionality")

    print("\n--- Benchmark: Nested Loop vs Vectorization ---")
    bench = benchmark_nested_vs_vectorized(size=100)
    print(f"Nested loop:  {bench['nested_loop_ms']:.2f} ms")
    print(f"Vectorized:   {bench['vectorized_ms']:.2f} ms")
    print(f"Speedup:      {bench['speedup']:.0f}x nhanh hơn")

    print("\n" + "=" * 60)
    print("LAB 11: Fine-Tuning Demo (Gemini)")
    print("=" * 60)

    print("\n--- Tạo Training Dataset ---")
    training_data = generate_training_dataset()
    print(f"Số training examples: {len(training_data)}")

    print("\n--- Mẫu Training Example ---")
    sample = training_data[2]
    print(json.dumps(sample, indent=2, ensure_ascii=False))

    print("\n--- Lưu JSONL File ---")
    save_training_jsonl(training_data, "training_data.jsonl")

    print("\n--- Kiểm tra file ---")
    with open("training_data.jsonl", 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(f"File có {len(lines)} dòng (mỗi dòng = 1 training example)")
    
    print("\n--- Khởi chạy Fine-Tuning Job ---")
    print("(Cần OpenAI API key với credits để chạy thực tế)")
    job_id = run_finetuning_job("training_data.jsonl")
    print(f"Job ID: {job_id}")
    
    print("\n--- Sau khi fine-tuning hoàn tất ---")
    print("Dùng model ID mới trong code:")
    print("  model = 'ft:gpt-4o-mini-2024-07-18:your-org::job-id'")
    print("Kết quả: code không còn comments thừa, gọn và sạch hơn base model")


2026-03-16 20:17:44,677 - __main__ - INFO - Computing L1 distance


LAB 10: Code Refactoring Demo

--- Test cả hai phiên bản ---
Info: computing L1 distance...
BEFORE (L1): 16.0
AFTER  (L1): 16.0
✓ Kết quả giống nhau – refactoring không thay đổi functionality

--- Benchmark: Nested Loop vs Vectorization ---
Nested loop:  3.22 ms
Vectorized:   0.11 ms
Speedup:      28x nhanh hơn

LAB 11: Fine-Tuning Demo (Gemini)

--- Tạo Training Dataset ---
Số training examples: 15

--- Mẫu Training Example ---
{
  "text_input": "You are a Python code assistant. When given a function signature, implement it with clean code only. No explanations, no inline comments, no docstrings. Return only the implementation.\n\nFUNCTION: {{ def is_palindrome(s: str) -> bool: }}\nCODE:",
  "output": "def is_palindrome(s: str) -> bool:\n    s = s.lower().replace(' ', '')\n    return s == s[::-1]"
}

--- Lưu JSONL File ---
Saved 15 training examples to training_data.jsonl

--- Kiểm tra file ---
File có 15 dòng (mỗi dòng = 1 training example)

--- Khởi chạy Gemini Tuning Job ---
(Cần c